# 09. 多 Agent 系統

學習如何建構多個 Agent 協作的系統。

---

## 🎯 學習目標

完成本章節後，您將能夠：
- ✅ 理解多 Agent 協作模式
- ✅ 實現 Supervisor 模式
- ✅ 建立專家團隊系統
- ✅ 設計 Agent 之間的通訊機制

---

## 📊 多 Agent 架構模式

### 模式 1: Supervisor 模式

```
┌─────────────────────────────────────────────────────────┐
│                   Supervisor 模式                        │
├─────────────────────────────────────────────────────────┤
│                                                         │
│                   ┌─────────────┐                       │
│                   │  Supervisor │                       │
│                   │   (主管)     │                       │
│                   └──────┬──────┘                       │
│                          │                              │
│            ┌─────────────┼─────────────┐                │
│            │             │             │                │
│            ▼             ▼             ▼                │
│     ┌──────────┐  ┌──────────┐  ┌──────────┐           │
│     │ Agent A  │  │ Agent B  │  │ Agent C  │           │
│     │ (研究員)  │  │ (作家)   │  │ (審核員) │           │
│     └──────────┘  └──────────┘  └──────────┘           │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### 模式 2: 專家團隊模式

```
┌─────────────────────────────────────────────────────────┐
│                   專家團隊模式                           │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐         │
│   │ 技術專家 │ ─▶ │ 商業專家 │ ─▶ │ 法務專家 │         │
│   └──────────┘    └──────────┘    └──────────┘         │
│         │              │              │                 │
│         └──────────────┴──────────────┘                 │
│                        │                                │
│                        ▼                                │
│                 ┌──────────────┐                        │
│                 │  綜合意見    │                        │
│                 └──────────────┘                        │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

In [1]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

---

## 9.1 Supervisor 模式

### 核心概念

| 角色 | 職責 |
|------|------|
| Supervisor | 分配任務、決定下一步 |
| Worker Agents | 執行特定任務 |
| 共享狀態 | 所有 Agent 共享同一個 State |

In [2]:
class TeamState(TypedDict):
    """團隊狀態"""
    messages: Annotated[list, add_messages]  # 對話歷史
    next_agent: str      # 下一個執行的 Agent
    task_complete: bool  # 任務是否完成
    iterations: int      # 迭代次數

def supervisor(state: TeamState) -> dict:
    """主管：決定下一個執行的 Agent
    
    模擬決策邏輯：
    - 迭代 0: 分配給研究員
    - 迭代 1: 分配給作家
    - 迭代 2+: 完成
    """
    iterations = state.get("iterations", 0)
    
    if iterations == 0:
        print("  👔 主管: 先讓研究員收集資料")
        return {"next_agent": "researcher", "iterations": 1}
    elif iterations == 1:
        print("  👔 主管: 讓作家撰寫內容")
        return {"next_agent": "writer", "iterations": 2}
    else:
        print("  👔 主管: 任務完成！")
        return {"next_agent": "FINISH", "task_complete": True, "iterations": iterations + 1}

def researcher(state: TeamState) -> dict:
    """研究員：負責搜尋和整理資訊"""
    print("  🔍 研究員: 正在收集資料...")
    return {"messages": [{"role": "assistant", "content": "[研究員] 已完成資料收集"}]}

def writer(state: TeamState) -> dict:
    """作家：負責撰寫內容"""
    print("  ✍️ 作家: 正在撰寫內容...")
    return {"messages": [{"role": "assistant", "content": "[作家] 已完成內容撰寫"}]}

print("✅ Agent 定義完成")

✅ Agent 定義完成


In [3]:
def route_agent(state: TeamState) -> Literal["researcher", "writer", "end"]:
    """路由函數：根據主管決策分配任務"""
    if state.get("task_complete"):
        return "end"
    
    agent = state.get("next_agent", "")
    if "research" in agent:
        return "researcher"
    elif "writ" in agent:
        return "writer"
    return "end"

# 建構團隊圖
graph = StateGraph(TeamState)

# 添加節點
graph.add_node("supervisor", supervisor)
graph.add_node("researcher", researcher)
graph.add_node("writer", writer)

# 定義流程
graph.add_edge(START, "supervisor")
graph.add_conditional_edges("supervisor", route_agent, {
    "researcher": "researcher",
    "writer": "writer",
    "end": END
})
graph.add_edge("researcher", "supervisor")  # 回報主管
graph.add_edge("writer", "supervisor")      # 回報主管

team_app = graph.compile()
print("✅ 多 Agent 團隊已就緒")

✅ 多 Agent 團隊已就緒


In [4]:
print("📊 圖結構:")
print(team_app.get_graph().draw_mermaid())

📊 圖結構:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	writer(writer)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	researcher --> supervisor;
	supervisor -. &nbsp;end&nbsp; .-> __end__;
	supervisor -.-> researcher;
	supervisor -.-> writer;
	writer --> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [5]:
print("🚀 執行多 Agent 團隊：")
print("=" * 50)

result = team_app.invoke({
    "messages": [{"role": "user", "content": "寫一篇關於 AI 發展的短文"}],
    "next_agent": "",
    "task_complete": False,
    "iterations": 0
})

print("\n" + "=" * 50)
print(f"📋 任務完成: {result['task_complete']}")
print(f"📊 總迭代: {result['iterations']}")
print(f"💬 訊息數: {len(result['messages'])}")

🚀 執行多 Agent 團隊：
  👔 主管: 先讓研究員收集資料
  🔍 研究員: 正在收集資料...
  👔 主管: 讓作家撰寫內容
  ✍️ 作家: 正在撰寫內容...
  👔 主管: 任務完成！

📋 任務完成: True
📊 總迭代: 3
💬 訊息數: 3


---

## 9.2 專家團隊模式

多個專家依序給出意見，最後綜合：

In [6]:
class ExpertState(TypedDict):
    """專家團隊狀態"""
    question: str                    # 問題
    expert_opinions: dict            # 各專家意見
    final_answer: str                # 最終答案

def tech_expert(state: ExpertState) -> dict:
    """技術專家"""
    print("  🖥️ 技術專家分析中...")
    opinion = f"技術觀點: 針對 '{state['question'][:20]}...' 的技術分析"
    return {"expert_opinions": {"tech": opinion}}

def business_expert(state: ExpertState) -> dict:
    """商業專家"""
    print("  💼 商業專家分析中...")
    opinions = state.get("expert_opinions", {}).copy()
    opinions["business"] = "商業觀點: 市場可行性分析"
    return {"expert_opinions": opinions}

def legal_expert(state: ExpertState) -> dict:
    """法務專家"""
    print("  ⚖️ 法務專家分析中...")
    opinions = state.get("expert_opinions", {}).copy()
    opinions["legal"] = "法務觀點: 合規性評估"
    return {"expert_opinions": opinions}

def synthesize(state: ExpertState) -> dict:
    """綜合所有專家意見"""
    print("  📝 綜合專家意見...")
    opinions = state.get("expert_opinions", {})
    summary = "綜合意見:\n"
    for expert, opinion in opinions.items():
        summary += f"  • {expert}: {opinion}\n"
    return {"final_answer": summary}

# 建構專家團隊
expert_graph = StateGraph(ExpertState)
expert_graph.add_node("tech", tech_expert)
expert_graph.add_node("business", business_expert)
expert_graph.add_node("legal", legal_expert)
expert_graph.add_node("synthesize", synthesize)

# 依序執行
expert_graph.add_edge(START, "tech")
expert_graph.add_edge("tech", "business")
expert_graph.add_edge("business", "legal")
expert_graph.add_edge("legal", "synthesize")
expert_graph.add_edge("synthesize", END)

expert_app = expert_graph.compile()
print("✅ 專家團隊已就緒")

✅ 專家團隊已就緒


In [7]:
print("🚀 執行專家團隊分析：")
print("=" * 50)

result = expert_app.invoke({
    "question": "企業應該如何採用生成式 AI？",
    "expert_opinions": {},
    "final_answer": ""
})

print("\n" + "=" * 50)
print("📋 最終答案:")
print(result["final_answer"])

🚀 執行專家團隊分析：
  🖥️ 技術專家分析中...
  💼 商業專家分析中...
  ⚖️ 法務專家分析中...
  📝 綜合專家意見...

📋 最終答案:
綜合意見:
  • tech: 技術觀點: 針對 '企業應該如何採用生成式 AI？...' 的技術分析
  • business: 商業觀點: 市場可行性分析
  • legal: 法務觀點: 合規性評估



---

## 💡 重點回顧

### 多 Agent 模式比較

| 模式 | 特點 | 適用場景 |
|------|------|----------|
| Supervisor | 中央控制、動態分配 | 複雜任務分解 |
| 專家團隊 | 串行執行、各司其職 | 多角度分析 |
| 對等協作 | 無中心、相互溝通 | 辯論、協商 |

### 設計要點

1. **清晰職責**：每個 Agent 職責明確
2. **共享狀態**：通過 State 傳遞資訊
3. **終止條件**：防止無限循環
4. **錯誤處理**：單個 Agent 失敗的處理

---

## 📝 練習題

1. **新增角色**：加入「審核員」Agent 進行品質檢查
2. **辯論機制**：讓兩個 Agent 針對問題辯論
3. **投票系統**：多個 Agent 投票決定結果
4. **錯誤恢復**：當某個 Agent 失敗時切換到備用

---

下一步：[10. 錯誤處理](10_error_handling.ipynb)